# Public reproducibility copy

This notebook is copied from the authoritative thesis evidence. Machine-specific paths were replaced by documented placeholders; scientific code, parameters, and reported values were not changed. External datasets, lexicons, and model files are not bundled.


# Evidence Notebook - Tuned RALSR Ablation

Recovery notebook for the frozen 53,130-configuration training grid. The search is not rerun; the complete grid and frozen selection artifacts are verified.

In [1]:
from pathlib import Path
import hashlib, json
import pandas as pd
from IPython.display import display, Markdown

CC = Path(r"${THESIS_CONTROL_ROOT}")
WS = Path(r"${THESIS_WORKSPACE}")
SUMMARY_PATH = Path(r"${THESIS_CONTROL_ROOT}\Evidence_Recovery_and_Thesis_Assets") / "06_Validation" / "Notebook_Summaries" / "Evidence_Notebook_Tuned_RALSR_Ablation_summary.json"
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest().upper()

print("Evidence notebook mode: frozen-artifact verification only")
print("Correction Control:", CC)
print("Writing Workspace:", WS)


Evidence notebook mode: frozen-artifact verification only
Correction Control: ${THESIS_CONTROL_ROOT}
Writing Workspace: ${THESIS_WORKSPACE}


## Fixed baseline, grid definition, and grid-size verification

In [2]:
d = CC / "Tuned_RALSR_Ablation"
grid_path = d / "TUNED_RALSR_GRID_RESULTS.csv"
weights_path = d / "TUNED_RALSR_WEIGHTS.json"
comparison_path = d / "TUNED_RALSR_TRAIN_DEV_COMPARISON.csv"
grid = pd.read_csv(grid_path)
weights = json.loads(weights_path.read_text(encoding="utf-8"))
print("Grid SHA256:", sha256(grid_path)); print("Weights SHA256:", sha256(weights_path))
assert len(grid) == 53130 and grid.Configuration_ID.nunique() == 53130
wcols = [c for c in grid.columns if c.endswith("_Weight")]
assert len(wcols) == 6
assert ((grid[wcols] * 20).round() == grid[wcols] * 20).all().all()
assert (grid[wcols].sum(axis=1).sub(1.0).abs() < 1e-9).all()
print("Grid: 53,130 unique nonnegative 0.05-simplex configurations - PASS")

Grid SHA256: 534DDD51F77C024A05325B707CEB44432ABDA8C45EDBE8ECFF4B68E5339EFD7B
Weights SHA256: 06C2447FABEA838DDBC09153B126EBCD4B23D1872DCCE35673966D1AA26D9F6B
Grid: 53,130 unique nonnegative 0.05-simplex configurations - PASS


## Selected configuration and fixed configuration position

In [3]:
selected = weights["selected_weights"]
print("Selected configuration:", weights["selected_configuration_id"])
display(pd.DataFrame({"Feature":weights["feature_order"],"Tuned weight":[selected[x] for x in weights["feature_order"]],"Fixed weight":[0.30,0.25,0.20,0.15,0.05,0.05]}))
assert weights["selected_configuration_id"] == "G01079"
fixed_vector = [0.30,0.25,0.20,0.15,0.05,0.05]
mask = pd.Series(True, index=grid.index)
for c,v in zip(wcols,fixed_vector): mask &= grid[c].sub(v).abs().lt(1e-12)
assert mask.sum() == 1
fixed_row = grid[mask].iloc[0]
display(fixed_row.to_frame("Fixed-grid record"))

Selected configuration: G01079


,Feature,Tuned weight,Fixed weight
0,Root_Coverage,0.00,0.30
1,Root_Overlap,0.00,0.25
2,Semantic_Overlap,0.25,0.20
3,Jaccard_Root_Similarity,0.55,0.15
4,Passage_Root_Count,0.10,0.05
5,Semantic_Count,0.10,0.05


,Fixed-grid record
Configuration_ID,G44028
Root_Coverage_Weight,0.3
Root_Overlap_Weight,0.25
Semantic_Overlap_Weight,0.2
Jaccard_Root_Similarity_Weight,0.15
Passage_Root_Count_Weight,0.05
Semantic_Count_Weight,0.05
Train_MAP_at_10,0.100846
Train_MRR_at_10,0.154764
L1_Distance_to_Fixed,0.0


## Frozen train and development comparison

In [4]:
comparison = pd.read_csv(comparison_path)
display(comparison)
assert comparison.loc[1,"Train_MAP_Delta_vs_Fixed"] > 0
assert comparison.loc[1,"Dev_MAP_Delta_vs_Fixed"] < 0

,Condition,Root_Coverage_Weight,Root_Overlap_Weight,Semantic_Overlap_Weight,Jaccard_Root_Similarity_Weight,Passage_Root_Count_Weight,Semantic_Count_Weight,Train_MAP_at_10,Train_MRR_at_10,Dev_MAP_at_10,Dev_MRR_at_10,Train_MAP_Delta_vs_Fixed,Train_MRR_Delta_vs_Fixed,Dev_MAP_Delta_vs_Fixed,Dev_MRR_Delta_vs_Fixed
0,Fixed heuristic RALSR,0.3,0.25,0.20,0.15,0.05,0.05,0.100846,0.154764,0.067381,0.126667,0.000000,0.000000,0.000000,0.000000
1,Training-selected tuned RALSR,0.0,0.00,0.25,0.55,0.10,0.10,0.106831,0.172526,0.067262,0.110000,0.005985,0.017761,-0.000119,-0.016667


## Held-out test comparison and generalization interpretation

Held-out values are loaded from the already-frozen Step-7 evaluation; no post-test selection occurs.

In [5]:
core = pd.read_csv(CC / "Final_Core_Evaluation" / "CORE_RESULTS_OFFICIAL_JUDGED.csv")
test = core[(core.Split == "test") & core.System.isin(["Fixed RALSR","Tuned RALSR"])][["System","MAP_at_10","MRR_at_10"]]
display(test)
summary = {
 "grid_size":len(grid),"selected_configuration":"G01079",
 "fixed_grid_rank":int(fixed_row["Selection_Rank"]),
 "tuned_train_map":float(comparison.loc[1,"Train_MAP_at_10"]),
 "fixed_train_map":float(comparison.loc[0,"Train_MAP_at_10"]),
 "tuned_dev_map":float(comparison.loc[1,"Dev_MAP_at_10"]),
 "fixed_dev_map":float(comparison.loc[0,"Dev_MAP_at_10"]),
 "tuned_test_map":float(test.loc[test.System.eq("Tuned RALSR"),"MAP_at_10"].iloc[0]),
 "fixed_test_map":float(test.loc[test.System.eq("Fixed RALSR"),"MAP_at_10"].iloc[0]),
 "substantive_mismatches":0,
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Training improves; development and held-out test MAP do not improve. Fixed RALSR remains primary.")
print("Substantive mismatch count: 0")

,System,MAP_at_10,MRR_at_10
8,Fixed RALSR,0.042897,0.135154
11,Tuned RALSR,0.040844,0.135862


Training improves; development and held-out test MAP do not improve. Fixed RALSR remains primary.
Substantive mismatch count: 0


## Thesis references

Use the Step-12 fixed-versus-tuned generalization figure and tuned-RALSR ablation table. The tuned system remains a diagnostic ablation.